# Role C (evaluation): measure OCR and the diacritic corrector

Needs the Colab secret `GITHUB_TOKEN` with **Notebook access** switched on for this notebook.

**First:** choose File > Save a copy in Drive and work in the copy, so the notebook in the shared folder stays clean. In the copy, switch on Notebook access for `GITHUB_TOKEN` again. Branch: `main`, which must contain role B's `src/corrector` (cell 1 stops with a message if it does not; then set `BRANCH` to the branch that has it).

What this notebook does, in order:

1. Connect to Drive and GitHub.
2. Settings that must be decided **before** looking at numbers.
3. Load the data and check it (coverage of the val split, labels, NFC, passages).
4. Raw OCR quality: CER and WER, with the raw text and with whitespace normalized, side by side.
5. Error classes and the hypothesis gate (diacritic share and benefit ceiling).
   5b. Exploratory, post hoc: what a syllable-level diacritic corrector could fix. It runs only after the gate thresholds are set and never changes the gate verdict.
   5c. Character confusions: runs `scripts/print_confusions.py` on val and saves the counts as JSON. It also runs only after the gate thresholds are set.
   5d. Exact ceiling, post hoc: the best CER and WER that any diacritic-only corrector could reach, computed by stripping diacritics on both sides. Sensitivity check of the gate, it does not change the gate verdict.
6. The corrector and the contract check. 6a loads role B's real corrector (a FAKE one is still built in for pipeline tests, `USE_FAKE = True`). 6b shows the distribution of its confidence.
7. Fixed / broken table, counted two ways: Option 1 (lines with the same syllable count) and Option 2 (word alignment). 7b: examples of fixed and broken syllables. 7c: check role A's `ocr_conf.jsonl` and test whether the OCR confidence separates right from wrong syllables.
8. Sweep of the confidence threshold tau on val, with two caps on the broken rate (1% and 2%).
9. Bootstrap by passage (main) and by page (val only, for comparison). 9b: compare the corrector levels on val (contract, determinism, calibration, best tau per level, paired bootstrap, level suggestion by a rule written before the numbers).
10. Score the test split. Off by default, runs once.
11. Save the val results to Drive.

Rules for reading the output:
- Numbers computed from a **draft OCR file** or from the **FAKE corrector** only check that the pipeline runs. They are not results. Output files get a `_DRAFT` and/or `_FAKE` suffix, and the test cell refuses to run while either is active.
- The gate thresholds in cell 2 are decided first. Until they are set, cell 5 does not show the diacritic share.
- Tau is chosen on val only. The test split is scored once, after tau **and the corrector level** are locked.
- Order for the real run: sections 1 to 9b with `CORRECTOR_LEVEL = None` and `FINAL_CAP = None`; read 9b and the cap comparison in section 8; set `FINAL_CAP` and `CORRECTOR_LEVEL` in the settings; run again from 6a to lock tau; only then set `RUN_TEST = True`.

## 1. Connect to Drive and GitHub

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

# The real corrector (role B) lives in src/corrector. Use "main" once its PR is merged;
# otherwise use the branch that has it (for example "b/corrector", or "c/demo" after merging main into it).
BRANCH = "main"

token = userdata.get("GITHUB_TOKEN").strip()
!rm -rf /content/repo
!git clone -q https://{token}@github.com/wickyhien18/Vietnamese-Handwriting-OCR.git /content/repo
!git -C /content/repo checkout -q {BRANCH}
%cd /content/repo
!pip install -q -r requirements.txt

import os
import sys

REPO_ROOT = "/content/repo"
sys.path.insert(0, REPO_ROOT)

DRIVE_ROOT = "/content/drive/MyDrive/Vietnamese-Handwriting-OCR"
DATA_DIR = f"{DRIVE_ROOT}/data"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
OUT_DIR = f"{DRIVE_ROOT}/outputs"

assert os.path.exists(f"{REPO_ROOT}/src/corrector/api.py"), (
    f"src/corrector/api.py is not on branch {BRANCH!r}: ask role B for the PR number, or set BRANCH to the branch that has it")

## 2. Settings decided before looking at numbers

Fill in the gate thresholds **before** the diacritic share is shown. Cell 5 stays hidden while they are `None`.

In [ ]:
OCR_OUT_PATH = f"{OUT_DIR}/ocr_out.jsonl"

# True while ocr_out.jsonl is a draft (for example a short training run).
# Set to False only when the real OCR output is in place. It adds "_DRAFT" to output names.
OCR_IS_DRAFT = False  # the OCR output is final (role A, 18/11/2026); True would mark every output _DRAFT

# --- Hypothesis gate (None = not decided yet; the values below were decided by the team, see the dev log) ---
# P_MIN: minimum share of diacritic errors among all errors, for example 0.5
# CEILING_MIN: minimum benefit ceiling (CER * share), for example 0.03
P_MIN = 0.5
CEILING_MIN = 0.03
SMOKE_LINES = 20  # lines used for the hidden smoke test while the gate is not set

# --- Corrector: role B's real one (src/corrector). Cell 6a loads it. ---
USE_FAKE = False  # True = built-in FAKE corrector (pipeline tests only)
CORRECTOR_MODULE = "src.corrector.api"  # one entry point: correct(text) and get_corrector(level)
CORRECTOR_LEVEL = None  # None = the default correct(); or "frequency", "bigram", "tagger". Choose it AFTER reading 9b.
CORRECTOR_LEVELS = ["frequency", "bigram", "tagger"]  # compared on val in 9b; edit to match role B's names
CORRECTOR_CKPT_ENV = "CORRECTOR_CKPT_DIR"  # a guess; 6a prints the variable names the code really reads
REAL_CORRECT_FN = None  # 6a sets this

# --- Choosing the level (rule decided before looking at any number from the real corrector) ---
# At FINAL_CAP take the level with the lowest val CER. Levels within TIE_POINTS CER points of the best are tied,
# and a tie goes to the level that comes first in SIMPLER_FIRST (simplest first).
SIMPLER_FIRST = ["frequency", "bigram", "tagger"]
TIE_POINTS = 0.05

# --- Choosing tau on val ---
CAPS = [0.01, 0.02]  # caps on the broken rate, compared side by side
FINAL_CAP = None  # after comparing both caps on val, set this to 0.01 or 0.02
TAU_GRID = [round(step * 0.05, 2) for step in range(21)]

# --- Bootstrap ---
N_BOOT = 1000
SEED = 42

# --- Test split (scored once) ---
RUN_TEST = False

## 3. Load the data and check it

The loader finds `contract_checks.py` and `classify_errors.py` anywhere in the repo, so a different folder layout does not break this cell. Any failed check stops the notebook with a message.

In [ ]:
import collections
import glob
import importlib.util
import json
import os
import random
import re
import unicodedata

from src.common.text_utils import normalize_nfc, strip_diacritics
from src.eval.align import DEL, INS, MATCH, SUB, align, corpus_error_rate, edit_distance


def load_repo_module(file_name):
    hits = sorted(
        path for path in glob.glob(f"{REPO_ROOT}/**/{file_name}", recursive=True)
        if "/.git/" not in path
    )
    assert hits, f"{file_name} was not found under {REPO_ROOT}"
    spec = importlib.util.spec_from_file_location(file_name[:-3], hits[0])
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    print("loaded", hits[0])
    return module


def read_jsonl(path):
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def read_json(path):
    with open(path, encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=1, default=lambda value: value.item() if hasattr(value, "item") else str(value))
    print("wrote", path)


contract_checks = load_repo_module("contract_checks.py")
classify_errors = load_repo_module("classify_errors.py")

index_records = read_jsonl(f"{OUT_DIR}/dataset_index.jsonl")
splits = read_json(f"{OUT_DIR}/splits.json")
passages = read_json(f"{OUT_DIR}/passages.json")
ocr_rows = read_jsonl(OCR_OUT_PATH)

problems = contract_checks.check_dataset_index_records(index_records)
problems += contract_checks.check_splits(splits, [rec["id"] for rec in index_records])
assert not problems, "\n".join(problems[:10])
print("index and splits pass the contract checks")

gt_by_id = {rec["id"]: rec["gt_text"] for rec in index_records}
split_sizes = collections.Counter(splits.values())
print("split sizes:", dict(split_sizes))

PAGE_PATTERN = re.compile(r"(?:train_data|test_data)/(\d+)/")
page_to_passage = {}
for passage_id, info in passages["passages"].items():
    for page in info["pages"]:
        if page not in info["excluded_pages"]:
            page_to_passage[page] = passage_id
passage_role = {passage_id: info["role"] for passage_id, info in passages["passages"].items()}


def build_lines(rows, split_name):
    """Turn the ocr_out rows of one split into line records, after strict checks."""
    expected_ids = {rec_id for rec_id, name in splits.items() if name == split_name}
    rows = [row for row in rows if row["split"] == split_name]
    ids = [row["id"] for row in rows]
    assert len(ids) == len(set(ids)), "duplicate ids in ocr_out"
    missing = sorted(expected_ids - set(ids))
    extra = sorted(set(ids) - expected_ids)
    assert not missing and not extra, (
        f"coverage mismatch for {split_name}: {len(missing)} missing {missing[:3]}, "
        f"{len(extra)} extra {extra[:3]}"
    )
    lines = []
    for row in rows:
        gt = gt_by_id[row["id"]]
        assert row["gt_text"] == gt, f"gt_text differs from the index for {row['id']}"
        ocr_raw = row["ocr_text"]
        assert ocr_raw == unicodedata.normalize("NFC", ocr_raw), f"ocr_text is not NFC for {row['id']}"
        match = PAGE_PATTERN.search(row["image_path"])
        assert match, f"cannot read the page number from {row['image_path']}"
        page = match.group(1)
        passage = page_to_passage.get(page)
        assert passage is not None, f"page {page} is not in any passage (excluded page?)"
        assert passage_role[passage] == split_name, f"page {page} belongs to a {passage_role[passage]} passage"
        lines.append({
            "id": row["id"],
            "gt": gt,
            "ocr_raw": ocr_raw,
            "ocr_norm": " ".join(ocr_raw.split()),
            "page": page,
            "passage": passage,
        })
    return lines


val_lines = build_lines(ocr_rows, "val")
print("val lines:", len(val_lines))
print("val passages:", len({line["passage"] for line in val_lines}), "| val pages:", len({line["page"] for line in val_lines}))

# line counts per passage must match passages.json (val passages have no excluded pages)
per_passage = collections.Counter(line["passage"] for line in val_lines)
for passage_id, count in sorted(per_passage.items()):
    expected = passages["passages"][passage_id]["lines"]
    assert count == expected, f"{passage_id}: {count} lines in ocr_out, {expected} in passages.json"
print("line counts per passage match passages.json")

irregular = sum(1 for line in val_lines if line["ocr_raw"] != line["ocr_norm"])
print("lines whose ocr_text has irregular whitespace: %d of %d (%.1f%%)" % (
    irregular, len(val_lines), 100.0 * irregular / len(val_lines)))

## 4. Raw OCR quality (CER and WER)

CER and WER are total edits divided by total reference length (not an average of per-line rates). The "normalized" columns collapse repeated spaces and trim the ends of the OCR text. The gap between the two (raw minus normalized) shows how much whitespace matters. It can be negative, because extra spaces in the OCR text sometimes line up with missing letters.

In [ ]:
_edit_cache = {}


def edits(ref, hyp):
    key = (ref, hyp)
    if key not in _edit_cache:
        _edit_cache[key] = edit_distance(ref, hyp)
    return _edit_cache[key]


def rate(pairs):
    """Pooled error rate: same definition as corpus_error_rate, with a cache."""
    total_ref = sum(len(ref) for ref, _ in pairs)
    return sum(edits(ref, hyp) for ref, hyp in pairs) / total_ref if total_ref else 0.0


def cer_wer(pairs):
    cer = rate(pairs)
    wer = rate([(tuple(ref.split()), tuple(hyp.split())) for ref, hyp in pairs])
    return cer, wer


pairs_raw = [(line["gt"], line["ocr_raw"]) for line in val_lines]
pairs_norm = [(line["gt"], line["ocr_norm"]) for line in val_lines]
assert abs(rate(pairs_raw) - corpus_error_rate(pairs_raw)) < 1e-12

cer_raw, wer_raw = cer_wer(pairs_raw)
cer_norm, wer_norm = cer_wer(pairs_norm)
print("%-22s %10s %10s" % ("", "CER", "WER"))
print("%-22s %9.2f%% %9.2f%%" % ("raw ocr_text", 100 * cer_raw, 100 * wer_raw))
print("%-22s %9.2f%% %9.2f%%" % ("whitespace normalized", 100 * cer_norm, 100 * wer_norm))
print("CER raw minus normalized: %.2f points (can be negative)" % (100 * (cer_raw - cer_norm)))

empty_lines = sum(1 for line in val_lines if not line["ocr_norm"])
same_count = sum(1 for line in val_lines if len(line["gt"].split()) == len(line["ocr_norm"].split()))
total_chars_gt = sum(len(line["gt"]) for line in val_lines)
total_chars_ocr = sum(len(line["ocr_norm"]) for line in val_lines)
print("empty OCR lines:", empty_lines, "of", len(val_lines))
print("lines with the same syllable count as the label: %d of %d (%.1f%%)" % (
    same_count, len(val_lines), 100.0 * same_count / len(val_lines)))
print("characters: label %d, OCR %d" % (total_chars_gt, total_chars_ocr))

per_line_cer = sorted(
    val_lines, key=lambda line: -edits(line["gt"], line["ocr_norm"]) / max(len(line["gt"]), 1))
print("\nfirst 3 lines:")
for line in val_lines[:3]:
    print("  GT :", line["gt"])
    print("  OCR:", repr(line["ocr_raw"]))
print("\n3 worst lines:")
for line in per_line_cer[:3]:
    print("  GT :", line["gt"])
    print("  OCR:", repr(line["ocr_raw"]))

## 5. Error classes and the hypothesis gate

`classify_errors.py` labels every character-level edit as `tone_mark`, `shape_mark`, `tone_and_shape`, `base_letter` or `spacing`. The diacritic share is (tone + shape + both) divided by **all** errors, spacing included. Benefit ceiling = CER x share: the most that a perfect diacritic corrector could remove.

While `P_MIN` or `CEILING_MIN` is `None` this cell only runs a hidden smoke test and prints no numbers.

In [ ]:
def classify_all(pairs):
    total = collections.Counter()
    for ref, hyp in pairs:
        total.update(classify_errors.classify_pair(ref, hyp))
    return total


GATE_READY = P_MIN is not None and CEILING_MIN is not None
gate_report = "not evaluated (thresholds not set)"

if not GATE_READY:
    classify_all(pairs_norm[:SMOKE_LINES])
    print("smoke test ok on %d lines. Numbers stay hidden until P_MIN and CEILING_MIN are set in cell 2." % SMOKE_LINES)
else:
    gate_report = {}
    for name, pairs, cer in (("raw", pairs_raw, cer_raw), ("normalized", pairs_norm, cer_norm)):
        counts = classify_all(pairs)
        share = classify_errors.diacritic_share(counts)
        ceiling = cer * share
        passed = share >= P_MIN and ceiling >= CEILING_MIN
        gate_report[name] = {
            "counts": dict(counts), "diacritic_share": share, "ceiling": ceiling, "passed": passed,
        }
        print("--- %s ---" % name)
        for label in classify_errors.ALL_LABELS:
            print("  %-16s %6d" % (label, counts[label]))
        print("  diacritic share: %.1f%% (minimum %.1f%%)" % (100 * share, 100 * P_MIN))
        print("  benefit ceiling: %.2f points (minimum %.2f)" % (100 * ceiling, 100 * CEILING_MIN))
        print("  gate:", "PASS" if passed else "FAIL")
    if gate_report["raw"]["passed"] != gate_report["normalized"]["passed"]:
        print("\nThe two versions disagree. Decide which one the gate uses and write it in the report.")

## 5b. Exploratory: what could a syllable-level diacritic corrector fix?

**This analysis is post hoc.** It was added after the gate result was known, so it never changes the gate verdict. Report it separately and label it exploratory.

The gate counts character edits. A corrector works on whole syllables, and one wrong diacritic makes a whole syllable wrong, so the share can look different at syllable level. Here the label and OCR syllables are aligned with `align.py` (the same alignment as WER), and every substituted syllable is sorted into two groups:
- **diacritic only:** the two syllables are equal once diacritics are removed (a diacritic corrector could fix it);
- **other:** the base letters differ (a diacritic corrector cannot fix it).

Missing and extra syllables are counted separately. The cell stops with a message while `P_MIN` or `CEILING_MIN` is `None`.

In [ ]:
exploratory_report = "not run"

if not (P_MIN is not None and CEILING_MIN is not None):
    print("Skipped: set P_MIN and CEILING_MIN in cell 2 first. This analysis is post hoc and must come after the gate.")
else:
    ops_total = collections.Counter()
    diacritic_pairs = collections.Counter()
    chars_removed = 0
    lines_exact_now = 0
    lines_exact_after_fix = 0
    for line in val_lines:
        gt_syllables = line["gt"].split()
        ocr_syllables = line["ocr_norm"].split()
        line_ops = collections.Counter()
        for kind, ref_syllable, hyp_syllable in align(gt_syllables, ocr_syllables):
            if kind == MATCH:
                line_ops["match"] += 1
            elif kind == SUB:
                if strip_diacritics(ref_syllable) == strip_diacritics(hyp_syllable):
                    line_ops["sub_diacritic_only"] += 1
                    diacritic_pairs[(ref_syllable, hyp_syllable)] += 1
                    chars_removed += edits(ref_syllable, hyp_syllable)
                else:
                    line_ops["sub_other"] += 1
            elif kind == INS:
                line_ops["ins"] += 1
            else:
                line_ops["del"] += 1
        ops_total.update(line_ops)
        wrong_without_diacritic_only = line_ops["sub_other"] + line_ops["ins"] + line_ops["del"]
        if wrong_without_diacritic_only == 0 and line_ops["sub_diacritic_only"] == 0:
            lines_exact_now += 1
        if wrong_without_diacritic_only == 0:
            lines_exact_after_fix += 1

    total_syllables = sum(len(line["gt"].split()) for line in val_lines)
    total_chars = sum(len(line["gt"]) for line in val_lines)
    diacritic_only = ops_total["sub_diacritic_only"]
    syllable_edits = diacritic_only + ops_total["sub_other"] + ops_total["ins"] + ops_total["del"]
    assert abs(syllable_edits / total_syllables - wer_norm) < 1e-12, "syllable edits do not match the WER of cell 4"

    wer_after_fix = (syllable_edits - diacritic_only) / total_syllables
    cer_ceiling_syllable = chars_removed / total_chars
    exploratory_report = {
        "syllables": total_syllables,
        "matched": ops_total["match"],
        "substituted_diacritic_only": diacritic_only,
        "substituted_other": ops_total["sub_other"],
        "missing": ops_total["del"],
        "extra": ops_total["ins"],
        "wer": syllable_edits / total_syllables,
        "share_of_syllable_edits_diacritic_only": diacritic_only / syllable_edits,
        "wer_ceiling_points": diacritic_only / total_syllables,
        "wer_if_all_diacritic_only_fixed": wer_after_fix,
        "cer_ceiling_from_diacritic_only_syllables": cer_ceiling_syllable,
        "lines_exact_now": lines_exact_now,
        "lines_exact_if_diacritic_only_fixed": lines_exact_after_fix,
        "lines": len(val_lines),
    }

    print("EXPLORATORY (post hoc), val, whitespace-normalized OCR")
    print("syllables in the labels: %d" % total_syllables)
    print("  matched:                          %6d" % ops_total["match"])
    print("  substituted, diacritic only:      %6d" % diacritic_only)
    print("  substituted, other (base letters):%6d" % ops_total["sub_other"])
    print("  missing from OCR:                 %6d" % ops_total["del"])
    print("  extra in OCR:                     %6d" % ops_total["ins"])
    print("WER from these counts: %.2f%% (same as cell 4)" % (100 * exploratory_report["wer"]))
    print("diacritic-only substitutions are %.1f%% of all syllable edits" % (100 * exploratory_report["share_of_syllable_edits_diacritic_only"]))
    print("WER ceiling: %.2f points (WER %.2f%% -> %.2f%% if every diacritic-only syllable were fixed)" % (
        100 * exploratory_report["wer_ceiling_points"], 100 * exploratory_report["wer"], 100 * wer_after_fix))
    print("CER ceiling counting only diacritic-only syllables: %.2f points" % (100 * cer_ceiling_syllable))
    for name in ("normalized",):
        if isinstance(gate_report, dict) and name in gate_report:
            print("for comparison, the gate (character level): share %.1f%%, ceiling %.2f points" % (
                100 * gate_report[name]["diacritic_share"], 100 * gate_report[name]["ceiling"]))
    print("lines that are exactly right: %d of %d now, %d if diacritic-only syllables were fixed" % (
        lines_exact_now, len(val_lines), lines_exact_after_fix))
    print("\nmost frequent diacritic-only substitutions (label -> OCR):")
    for (ref_syllable, hyp_syllable), count in diacritic_pairs.most_common(15):
        print("  %-12s -> %-12s %4d" % (ref_syllable, hyp_syllable, count))
    print("\nThis is exploratory. The gate verdict is unchanged; report both levels.")

## 5c. Character confusions (runs `scripts/print_confusions.py`)

This cell runs the stand-alone script `scripts/print_confusions.py` on the OCR file and prints the most frequent character confusions per error label, the missing and extra characters, and the least reliable label characters. The counts are also saved to `outputs/confusions_<split>.json`.

- Use `val` (or `train`) when the tables guide any tuning. **Never use the test split to tune anything.**
- The script must exist in the repo clone (push `scripts/print_confusions.py` to the branch of cell 1 and run cell 1 again). As a shortcut, upload the file to `/content` and this cell copies it into the clone.
- Like 5b, it stays hidden until `P_MIN` and `CEILING_MIN` are set, because the tables show the diacritic share.

In [ ]:
import shutil
import subprocess
import sys

CONFUSION_SPLIT = "val"  # never analyse the test split to tune anything
CONFUSION_TOP = 15
assert CONFUSION_SPLIT in ("train", "val"), "use val or train: the test split must not guide any tuning"

script_path = f"{REPO_ROOT}/scripts/print_confusions.py"
uploaded_copy = "/content/print_confusions.py"  # fallback: the file uploaded by hand to this Colab session

if not (P_MIN is not None and CEILING_MIN is not None):
    print("Skipped: set P_MIN and CEILING_MIN in cell 2 first. The confusion tables show the diacritic share, so they come after the gate.")
else:
    if not os.path.exists(script_path) and os.path.exists(uploaded_copy):
        os.makedirs(os.path.dirname(script_path), exist_ok=True)
        shutil.copy(uploaded_copy, script_path)
        print("copied", uploaded_copy, "to", script_path)
    if not os.path.exists(script_path):
        print("scripts/print_confusions.py was not found in the repo clone.")
        print("Push it to the branch used in cell 1 and run cell 1 again, or upload it to /content and run this cell again.")
    else:
        suffix = "_DRAFT" if OCR_IS_DRAFT else ""
        confusion_json = f"{OUT_DIR}/confusions_{CONFUSION_SPLIT}{suffix}.json"
        result = subprocess.run(
            [sys.executable, script_path, "--ocr-out", OCR_OUT_PATH, "--split", CONFUSION_SPLIT,
             "--top", str(CONFUSION_TOP), "--out-json", confusion_json],
            capture_output=True, text=True, cwd=REPO_ROOT,
        )
        print(result.stdout)
        if result.returncode != 0:
            print("print_confusions.py failed:")
            print(result.stderr)

## 5d. Exact ceiling of any diacritic-only corrector (post hoc)

The gate counts diacritic errors in **one** optimal alignment, but a pair of strings can have several optimal alignments, so that count can miss fixable errors. Example: label `a`, OCR `a'o` (an accented `a` followed by `o`; the accent is written as an apostrophe here). The alignment "insert the accented `a`, substitute `a` by `o`" has no diacritic error, yet a corrector that removes the accent (giving `ao`) removes one edit.

The exact ceiling does not depend on the alignment. For every text `h'` that differs from the OCR text `h` only in diacritics, `d(ref, h') >= d(strip(ref), strip(h))`, and the bound is reachable (replace each aligned pair that is equal after stripping by the label character). So the best CER of **any** diacritic-only corrector, even a perfect one, is the CER of the stripped texts. The same holds for WER with syllables as units.

This cell reports the exact ceiling and checks the gate again with it as a sensitivity analysis. It never changes the pre-registered gate verdict. It stays hidden until `P_MIN` and `CEILING_MIN` are set.

In [ ]:
exact_ceiling_report = "not run"

if not (P_MIN is not None and CEILING_MIN is not None):
    print("Skipped: set P_MIN and CEILING_MIN in cell 2 first. This analysis is post hoc and must come after the gate.")
else:
    ref_stripped = [strip_diacritics(line["gt"]) for line in val_lines]
    hyp_stripped = [strip_diacritics(line["ocr_norm"]) for line in val_lines]

    total_chars_exact = sum(len(line["gt"]) for line in val_lines)
    char_edits_now = sum(edits(line["gt"], line["ocr_norm"]) for line in val_lines)
    char_edits_stripped = sum(edits(ref, hyp) for ref, hyp in zip(ref_stripped, hyp_stripped))
    assert abs(char_edits_now / total_chars_exact - cer_norm) < 1e-12, "character edits do not match the CER of cell 4"

    total_words_exact = sum(len(line["gt"].split()) for line in val_lines)
    word_edits_now = sum(edits(tuple(line["gt"].split()), tuple(line["ocr_norm"].split())) for line in val_lines)
    word_edits_stripped = sum(edits(tuple(ref.split()), tuple(hyp.split())) for ref, hyp in zip(ref_stripped, hyp_stripped))
    assert abs(word_edits_now / total_words_exact - wer_norm) < 1e-12, "word edits do not match the WER of cell 4"

    cer_ceiling_exact = (char_edits_now - char_edits_stripped) / total_chars_exact
    wer_ceiling_exact = (word_edits_now - word_edits_stripped) / total_words_exact
    exact_share = (char_edits_now - char_edits_stripped) / char_edits_now if char_edits_now else 0.0
    exact_pass = exact_share >= P_MIN and cer_ceiling_exact >= CEILING_MIN

    exact_ceiling_report = {
        "cer_now": cer_norm,
        "cer_after_perfect_diacritic_corrector": char_edits_stripped / total_chars_exact,
        "cer_ceiling_points": cer_ceiling_exact,
        "exact_diacritic_share": exact_share,
        "wer_now": wer_norm,
        "wer_after_perfect_diacritic_corrector": word_edits_stripped / total_words_exact,
        "wer_ceiling_points": wer_ceiling_exact,
        "sensitivity_gate_passed": exact_pass,
    }

    print("EXACT CEILING (post hoc), val, whitespace-normalized OCR")
    print("CER now %.2f%% -> %.2f%% after a perfect diacritic corrector (ceiling %.2f points)" % (
        100 * cer_norm, 100 * char_edits_stripped / total_chars_exact, 100 * cer_ceiling_exact))
    print("WER now %.2f%% -> %.2f%% after a perfect diacritic corrector (ceiling %.2f points)" % (
        100 * wer_norm, 100 * word_edits_stripped / total_words_exact, 100 * wer_ceiling_exact))
    print("exact diacritic share of character edits: %.1f%%" % (100 * exact_share))
    if isinstance(gate_report, dict):
        print("for comparison, label counts in one alignment (the gate): share %.1f%%, ceiling %.2f points, gate %s" % (
            100 * gate_report["normalized"]["diacritic_share"], 100 * gate_report["normalized"]["ceiling"],
            "PASS" if gate_report["normalized"]["passed"] else "FAIL"))
    if isinstance(exploratory_report, dict):
        print("for comparison, syllable substitutions that differ only in diacritics (5b): WER ceiling %.2f points" % (
            100 * exploratory_report["wer_ceiling_points"]))
    print("sensitivity, gate with the exact numbers: share >= %.0f%%? %s | ceiling >= %.0f points? %s -> %s" % (
        100 * P_MIN, "yes" if exact_share >= P_MIN else "no",
        100 * CEILING_MIN, "yes" if cer_ceiling_exact >= CEILING_MIN else "no", "PASS" if exact_pass else "FAIL"))
    print("\nThis is a sensitivity analysis. The pre-registered gate verdict is unchanged; report both.")

## 6. The corrector

`correct(text)` must return `{"text": str, "conf": [float]}` with the same number of syllables, only diacritics changed, and one confidence per syllable. Every call is checked with `check_correct_output`.

The corrector always receives the **whitespace-normalized** OCR text, so any change comes from diacritics only.

**The built-in FAKE corrector peeks at the label of the current line** to produce a mix of fixed, kept and broken syllables. It exists only to test the pipeline. Anything computed with it is saved with a `_FAKE` suffix.

With `USE_FAKE = False` the corrector is role B's real one, loaded by cell 6a. The check below stops the notebook if any of the 636 val lines breaks the contract.

### 6a. Load role B's real corrector

Loads `src.corrector.api` (`correct` and `get_corrector(level)`), prints the environment variables its code reads (so the checkpoint folder is set by the real name, not a guess), prints what `contracts/corrector_api.md` says about levels, and records the repo commit. Edge cases from the contract (empty text, digits, punctuation, mixed case, unknown syllables) are tried first. A failure there is a warning for role B, not a stop: the 636 val lines are checked strictly in section 6.

Skipped while `USE_FAKE = True`.

In [ ]:
import importlib
import inspect
import subprocess
import time

CORRECTOR_DIR = f"{REPO_ROOT}/src/corrector"
CKPT_DEFAULT = f"{CKPT_DIR}/corrector"  # role B's model files: checkpoints/corrector/ on the shared Drive
EDGE_CASES = ["", "123", "Xin Chao cac ban", "ban oi , 2026 !", "zzqx vvwk", "NGUYEN van A", "a"]

GIT_HEAD = subprocess.run(
    ["git", "-C", REPO_ROOT, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip() or "unknown"
api = None
HAS_LEVELS = False


def env_names_read_by(package_dir):
    """Names of the environment variables that the corrector package reads (found by reading its source)."""
    pattern = re.compile(r"(?:environ(?:\.get|\.setdefault)?\s*[\[(]|getenv\s*\()\s*[\"']([A-Za-z0-9_]+)[\"']")
    names = set()
    for path in glob.glob(f"{package_dir}/**/*.py", recursive=True):
        with open(path, encoding="utf-8") as handle:
            names.update(pattern.findall(handle.read()))
    return sorted(names)


def get_level_fn(level):
    """level None = the default correct(); otherwise get_corrector(level)."""
    if level is None:
        return api.correct
    assert HAS_LEVELS, "get_corrector() is missing: ask role B for it (item 2 of the task list)"
    return api.get_corrector(level)


def probe_corrector(fn, name):
    """Edge cases from the contract. Returns the list of failures."""
    started = time.time()
    bad = []
    for text in EDGE_CASES:
        try:
            out = fn(text)
            problems = contract_checks.check_correct_output(text, out)
        except Exception as error:  # a crash is a contract violation too
            problems = ["raised %s: %s" % (type(error).__name__, error)]
        if problems:
            bad.append((text, problems[:2]))
    print("%-10s edge cases: %d of %d fail (%.1f s)" % (name, len(bad), len(EDGE_CASES), time.time() - started))
    for text, problems in bad:
        print("   %r -> %s" % (text, problems))
    return bad


if USE_FAKE:
    print("USE_FAKE is True: the real corrector is not loaded. Numbers are pipeline checks only.")
else:
    env_names = env_names_read_by(CORRECTOR_DIR)
    print("environment variables read in src/corrector:", env_names or "none found")
    ckpt_names = [name for name in env_names if "CKPT" in name.upper() or "CHECKPOINT" in name.upper()]
    for name in ckpt_names or [CORRECTOR_CKPT_ENV]:
        os.environ.setdefault(name, CKPT_DEFAULT)
        print("  %s = %s" % (name, os.environ[name]))
    if not ckpt_names:
        print("  no variable with CKPT in its name was found, so the code uses its own default path;")
        print("  if loading fails below, ask role B which variable or argument sets the checkpoint folder.")

    api = importlib.import_module(CORRECTOR_MODULE)
    print("\nloaded", api.__file__)
    assert hasattr(api, "correct"), f"{CORRECTOR_MODULE} has no correct()"
    HAS_LEVELS = hasattr(api, "get_corrector")
    print("correct", inspect.signature(api.correct))
    print("get_corrector", inspect.signature(api.get_corrector) if HAS_LEVELS
          else "MISSING: the level comparison (9b) will be unavailable")

    doc_path = f"{REPO_ROOT}/contracts/corrector_api.md"
    if os.path.exists(doc_path):
        with open(doc_path, encoding="utf-8") as handle:
            hits = [line.rstrip() for line in handle
                    if re.search(r"default|mặc định|get_corrector|level|mức", line, re.I)]
        print("\ncontracts/corrector_api.md, lines about levels (role B writes the default level and why):")
        for line in hits[:25]:
            print("  ", line)

    started = time.time()
    REAL_CORRECT_FN = get_level_fn(CORRECTOR_LEVEL)
    level_name = CORRECTOR_LEVEL or "default"
    print("\nlevel %r loaded in %.1f s | repo commit %s" % (level_name, time.time() - started, GIT_HEAD))
    edge_failures = probe_corrector(REAL_CORRECT_FN, level_name)
    if edge_failures:
        print("WARNING: edge-case failures do not block the measurement, but role B must fix them (item 4 asks for edge-case tests).")

In [ ]:
TONE_MARKS = ["\u0300", "\u0301", "\u0303", "\u0309", "\u0323"]
_fake_state = {"gt": None}


def change_tone(syllable, rng):
    """Put a different tone mark on a syllable (used by the FAKE corrector to break a syllable)."""
    decomposed = unicodedata.normalize("NFD", syllable)
    present = {ch for ch in decomposed if ch in TONE_MARKS}
    stripped = "".join(ch for ch in decomposed if ch not in TONE_MARKS)
    vowel_positions = [pos for pos, ch in enumerate(stripped) if ch in "aeiouyAEIOUY"]
    options = [mark for mark in TONE_MARKS if mark not in present]
    if not vowel_positions or not options:
        return syllable
    insert_at = vowel_positions[-1] + 1
    while insert_at < len(stripped) and unicodedata.combining(stripped[insert_at]):
        insert_at += 1
    mark = options[rng.randrange(len(options))]
    candidate = unicodedata.normalize("NFC", stripped[:insert_at] + mark + stripped[insert_at:])
    if strip_diacritics(candidate) != strip_diacritics(syllable):
        return syllable
    return candidate


def fake_correct(text):
    """FAKE corrector for pipeline tests. It looks at the label of the current line."""
    syllables = normalize_nfc(text).split()
    label_syllables = _fake_state["gt"].split() if _fake_state["gt"] else []
    use_label = len(label_syllables) == len(syllables)
    out, conf = [], []
    for position, syllable in enumerate(syllables):
        rng = random.Random(f"fake|{syllable}|{position}")
        chosen, confidence = syllable, rng.uniform(0.3, 1.0)
        if use_label:
            target = label_syllables[position]
            if target != syllable and strip_diacritics(target) == strip_diacritics(syllable):
                if rng.random() < 0.7:  # a fixable error: fix it most of the time
                    chosen, confidence = target, rng.uniform(0.5, 1.0)
                else:
                    confidence = rng.uniform(0.0, 0.6)
            elif target == syllable:
                if rng.random() < 0.1:  # an already right syllable: break it sometimes
                    chosen, confidence = change_tone(syllable, rng), rng.uniform(0.0, 0.6)
                else:
                    confidence = rng.uniform(0.6, 1.0)
        out.append(chosen)
        conf.append(round(confidence, 4))
    return {"text": " ".join(out), "conf": conf}


if USE_FAKE:
    CORRECT_FN = fake_correct
else:
    assert REAL_CORRECT_FN is not None, "run cell 6a to load the real corrector (REAL_CORRECT_FN), or set USE_FAKE = True"
    CORRECT_FN = REAL_CORRECT_FN

RESULT_SUFFIX = ("_DRAFT" if OCR_IS_DRAFT else "") + ("_FAKE" if USE_FAKE else "")


def banner():
    if OCR_IS_DRAFT:
        print("*** DRAFT OCR: numbers below only check the pipeline ***")
    if USE_FAKE:
        print("*** FAKE corrector: numbers below only check the pipeline ***")


def correct_lines(lines):
    for line in lines:
        text_in = line["ocr_norm"]
        if USE_FAKE:
            _fake_state["gt"] = line["gt"]
        out = CORRECT_FN(text_in)
        problems = contract_checks.check_correct_output(text_in, out)
        assert not problems, f"{line['id']}: {problems[:3]}"
        line["corr_syll"] = out["text"].split()
        line["conf"] = [float(value) for value in out["conf"]]
    _fake_state["gt"] = None


banner()
correct_lines(val_lines)
print("corrector output passes check_correct_output on all %d val lines" % len(val_lines))

### 6b. Distribution of the corrector's confidence

Tau is swept from 0.00 to 1.00 in steps of 0.05, so `conf` has to spread out. If almost all values sit near 0.99, every tau below 0.95 gives the same result and the sweep says nothing. The table shows how many of the syllables that the corrector **changes** survive each tau.

In [ ]:
import matplotlib.pyplot as plt

banner()


def quantiles(values, qs=(0.1, 0.25, 0.5, 0.75, 0.9)):
    ordered = sorted(values)
    return [ordered[min(int(q * len(ordered)), len(ordered) - 1)] for q in qs] if ordered else []


all_conf = [value for line in val_lines for value in line["conf"]]
changed_conf = [
    line["conf"][pos]
    for line in val_lines
    for pos, (corrected, original) in enumerate(zip(line["corr_syll"], line["ocr_norm"].split()))
    if corrected != original
]
print("syllables given to the corrector: %d | changed by it: %d (%.1f%%)" % (
    len(all_conf), len(changed_conf), 100.0 * len(changed_conf) / len(all_conf)))
print("conf quantiles (10/25/50/75/90%%), all syllables:     %s" % ", ".join("%.3f" % v for v in quantiles(all_conf)))
print("conf quantiles (10/25/50/75/90%%), changed syllables: %s" % ", ".join("%.3f" % v for v in quantiles(changed_conf)))
print("share of all syllables with conf >= 0.95: %.1f%%" % (100.0 * sum(1 for v in all_conf if v >= 0.95) / len(all_conf)))
if not changed_conf:
    print("The corrector changes nothing on val: there is nothing for tau to filter.")
else:
    print("\nchanged syllables that survive each tau:")
    for tau in (0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99):
        kept = sum(1 for v in changed_conf if v >= tau)
        print("  tau >= %.2f: %5d of %d (%.0f%%)" % (tau, kept, len(changed_conf), 100.0 * kept / len(changed_conf)))

figure, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].hist(all_conf, bins=20, range=(0, 1))
axes[0].set_title("conf, all val syllables")
axes[0].set_xlabel("conf")
if changed_conf:
    axes[1].hist(changed_conf, bins=20, range=(0, 1))
axes[1].set_title("conf, syllables the corrector changes")
axes[1].set_xlabel("conf")
plt.tight_layout()
plt.show()

conf_distribution = {
    "n_syllables": len(all_conf), "n_changed": len(changed_conf),
    "quantiles_all": quantiles(all_conf), "quantiles_changed": quantiles(changed_conf),
    "share_conf_ge_095": sum(1 for v in all_conf if v >= 0.95) / len(all_conf),
}

## 7. Fixed / broken table, counted two ways

For every syllable we compare three things: the label, the OCR output and the corrected output.

| | corrected is right | corrected is wrong |
|---|---|---|
| **OCR was right** | kept right | **broken** |
| **OCR was wrong** | **fixed** | still wrong |

Broken rate = broken / (kept right + broken). Net fixed = fixed - broken.

Syllables can only be compared position by position when the label and the OCR line have the same number of syllables. Two ways to handle the other lines:
- **Option 1:** use only the lines with the same syllable count (clean, but drops lines).
- **Option 2:** align label and OCR syllables with `align.py` and use the matched or substituted pairs (uses most syllables).

In [ ]:
def prepare_pairs(lines):
    """Store the syllable pairs of both options on every line."""
    for line in lines:
        gt = line["gt"].split()
        ocr = line["ocr_norm"].split()
        line["pairs1"] = [(pos, pos) for pos in range(len(gt))] if len(gt) == len(ocr) else []
        pairs2 = []
        ref_pos = hyp_pos = 0
        for kind, _, _ in align(gt, ocr):
            if kind in (MATCH, SUB):
                pairs2.append((ref_pos, hyp_pos))
                ref_pos += 1
                hyp_pos += 1
            elif kind == INS:
                hyp_pos += 1
            else:
                ref_pos += 1
        line["pairs2"] = pairs2


def final_syllables(line, tau):
    ocr = line["ocr_norm"].split()
    if tau is None:
        return ocr
    conf = line["conf"]
    corr = line["corr_syll"]
    return [corr[pos] if conf[pos] >= tau else ocr[pos] for pos in range(len(ocr))]


def final_text(line, tau):
    return " ".join(final_syllables(line, tau))


def four_cell(lines, option, tau):
    counts = {"kept_right": 0, "broken": 0, "fixed": 0, "still_wrong": 0}
    used_lines = 0
    for line in lines:
        pairs = line["pairs1"] if option == 1 else line["pairs2"]
        if not pairs:
            continue
        used_lines += 1
        gt = line["gt"].split()
        ocr = line["ocr_norm"].split()
        final = final_syllables(line, tau)
        for ref_pos, hyp_pos in pairs:
            ocr_right = ocr[hyp_pos] == gt[ref_pos]
            final_right = final[hyp_pos] == gt[ref_pos]
            if ocr_right and final_right:
                counts["kept_right"] += 1
            elif ocr_right:
                counts["broken"] += 1
            elif final_right:
                counts["fixed"] += 1
            else:
                counts["still_wrong"] += 1
    right = counts["kept_right"] + counts["broken"]
    counts["syllables"] = right + counts["fixed"] + counts["still_wrong"]
    counts["lines_used"] = used_lines
    counts["broken_rate"] = counts["broken"] / right if right else 0.0
    counts["net_fixed"] = counts["fixed"] - counts["broken"]
    return counts


prepare_pairs(val_lines)
banner()
total_ref_syllables = sum(len(line["gt"].split()) for line in val_lines)
table_tau0 = {1: four_cell(val_lines, 1, 0.0), 2: four_cell(val_lines, 2, 0.0)}

print("correct everything (tau = 0.00):")
print("%-26s %14s %14s" % ("", "Option 1", "Option 2"))
rows = [
    ("lines used", "lines_used"), ("syllables compared", "syllables"),
    ("kept right", "kept_right"), ("broken", "broken"), ("fixed", "fixed"),
    ("still wrong", "still_wrong"), ("net fixed", "net_fixed"),
]
for title, key in rows:
    print("%-26s %14d %14d" % (title, table_tau0[1][key], table_tau0[2][key]))
print("%-26s %13.2f%% %13.2f%%" % ("broken rate", 100 * table_tau0[1]["broken_rate"], 100 * table_tau0[2]["broken_rate"]))
print("share of lines dropped by Option 1: %.1f%%" % (100.0 * (1 - table_tau0[1]["lines_used"] / len(val_lines))))
print("share of label syllables used: Option 1 %.1f%%, Option 2 %.1f%%" % (
    100.0 * table_tau0[1]["syllables"] / total_ref_syllables,
    100.0 * table_tau0[2]["syllables"] / total_ref_syllables))

### 7b. Examples of what the corrector fixes and breaks (tau = 0, Option 2 pairs)

The table above counts. This cell shows. **Broken** examples are sorted by confidence, highest first: those are the mistakes that no tau can filter out. **Fixed** examples are sorted lowest confidence first: those are the fixes that a high tau would throw away. Each row shows the OCR syllable with two neighbours on each side.

In [ ]:
def collect_changes(lines):
    fixed, broken = [], []
    for line in lines:
        gt = line["gt"].split()
        ocr = line["ocr_norm"].split()
        for ref_pos, hyp_pos in line["pairs2"]:
            corrected = line["corr_syll"][hyp_pos]
            if corrected == ocr[hyp_pos]:
                continue
            context = " ".join(ocr[max(hyp_pos - 2, 0):hyp_pos + 3])
            entry = (line["conf"][hyp_pos], ocr[hyp_pos], corrected, gt[ref_pos], context)
            if corrected == gt[ref_pos]:
                fixed.append(entry)
            elif ocr[hyp_pos] == gt[ref_pos]:
                broken.append(entry)
    return fixed, broken


banner()
fixed_examples, broken_examples = collect_changes(val_lines)
print("fixed: %d | broken: %d (Option 2 pairs, tau = 0)" % (len(fixed_examples), len(broken_examples)))
print("\nbroken, highest confidence first (label = OCR, corrector changed it to something else):")
print("%-6s %-10s -> %-10s | label %-10s | context" % ("conf", "OCR", "corrected", ""))
for conf_value, ocr_syllable, corrected, label, context in sorted(broken_examples, reverse=True)[:10]:
    print("%-6.3f %-10s -> %-10s | label %-10s | %s" % (conf_value, ocr_syllable, corrected, label, context))
print("\nfixed, lowest confidence first:")
for conf_value, ocr_syllable, corrected, label, context in sorted(fixed_examples)[:10]:
    print("%-6.3f %-10s -> %-10s | label %-10s | %s" % (conf_value, ocr_syllable, corrected, label, context))

### 7c. Check `ocr_conf.jsonl` and test whether the OCR confidence tells right syllables from wrong ones

Role A's file holds, per line, `char` (one value per character of `ocr_text`, spaces included), `syllable_min` and `syllable_mean` (one value per syllable). First the hand-over checks from the dev log: `ocr_text` identical to `ocr_out.jsonl`, lengths, values in [0, 1]. Then, on the Option 2 pairs: AUROC of each syllable score for "syllable is right" against "syllable is wrong" and against "wrong only in diacritics", and the share of right syllables per confidence band.

This is diagnostic. The main flow gates on the **corrector's** `conf`, not on this one. A score is useful only if the AUROC is clearly above 0.5 and the bands rise from left to right. Skipped when the file is missing.

In [ ]:
OCR_CONF_PATH = f"{OUT_DIR}/ocr_conf.jsonl"
SCORE_KEYS = ("syllable_min", "syllable_mean")
BAND_EDGES = [0.0, 0.5, 0.7, 0.9, 0.95, 1.0000001]


def auroc(scores, labels):
    """P(score of a random positive > score of a random negative), ties count one half (Mann-Whitney)."""
    order = sorted(range(len(scores)), key=lambda index: scores[index])
    ranks = [0.0] * len(scores)
    start = 0
    while start < len(order):
        stop = start
        while stop + 1 < len(order) and scores[order[stop + 1]] == scores[order[start]]:
            stop += 1
        for position in range(start, stop + 1):
            ranks[order[position]] = (start + stop) / 2 + 1
        start = stop + 1
    positives = sum(1 for label in labels if label)
    negatives = len(labels) - positives
    if positives == 0 or negatives == 0:
        return float("nan")
    rank_sum = sum(rank for rank, label in zip(ranks, labels) if label)
    return (rank_sum - positives * (positives + 1) / 2) / (positives * negatives)


def check_ocr_conf():
    if not os.path.exists(OCR_CONF_PATH):
        print("Skipped: %s does not exist (role A hands it over)." % OCR_CONF_PATH)
        return "not run"
    conf_by_id = {row["id"]: row for row in read_jsonl(OCR_CONF_PATH)}
    ocr_by_id = {row["id"]: row for row in ocr_rows}
    for line in val_lines:
        row = conf_by_id[line["id"]]
        assert row["ocr_text"] == ocr_by_id[line["id"]]["ocr_text"], f"ocr_text differs from ocr_out for {line['id']}"
        n_syllables = len(line["ocr_raw"].split())
        assert len(row["char"]) == len(line["ocr_raw"]), (
            f"char has {len(row['char'])} values for {len(line['ocr_raw'])} characters ({line['id']})")
        assert len(row["syllable_min"]) == n_syllables and len(row["syllable_mean"]) == n_syllables, (
            f"syllable lists have the wrong length ({line['id']})")
        for key in ("char", "syllable_min", "syllable_mean"):
            assert all(0.0 <= value <= 1.0 for value in row[key]), f"{key} is outside [0, 1] ({line['id']})"
    print("ocr_conf.jsonl: ocr_text identical, lengths and ranges fine on all %d val lines" % len(val_lines))

    scores = {key: [] for key in SCORE_KEYS}
    is_right, is_diacritic_only_wrong = [], []
    for line in val_lines:
        row = conf_by_id[line["id"]]
        gt = line["gt"].split()
        ocr = line["ocr_norm"].split()
        for ref_pos, hyp_pos in line["pairs2"]:
            for key in SCORE_KEYS:
                scores[key].append(row[key][hyp_pos])
            right = ocr[hyp_pos] == gt[ref_pos]
            is_right.append(right)
            is_diacritic_only_wrong.append(
                (not right) and strip_diacritics(ocr[hyp_pos]) == strip_diacritics(gt[ref_pos]))

    report = {"syllables": len(is_right), "right_share": sum(is_right) / len(is_right), "auroc": {}, "bands": {}}
    keep = [i for i, (right, fixable) in enumerate(zip(is_right, is_diacritic_only_wrong)) if right or fixable]
    print("\nsyllables compared (Option 2): %d, right %.1f%%, wrong only in diacritics %d" % (
        len(is_right), 100 * report["right_share"], sum(is_diacritic_only_wrong)))
    print("%-14s %26s %38s" % ("score", "AUROC right vs any wrong", "AUROC right vs diacritic-only wrong"))
    for key in SCORE_KEYS:
        overall = auroc(scores[key], is_right)
        fixable_only = auroc([scores[key][i] for i in keep], [is_right[i] for i in keep])
        report["auroc"][key] = {"right_vs_wrong": overall, "right_vs_diacritic_only_wrong": fixable_only}
        print("%-14s %26.3f %38.3f" % (key, overall, fixable_only))
    for key in SCORE_KEYS:
        print("\n%s: share of syllables that are right, by confidence band" % key)
        report["bands"][key] = []
        for low, high in zip(BAND_EDGES[:-1], BAND_EDGES[1:]):
            members = [i for i, value in enumerate(scores[key]) if low <= value < high]
            right_share = sum(is_right[i] for i in members) / len(members) if members else float("nan")
            report["bands"][key].append({"low": low, "high": min(high, 1.0), "n": len(members), "right_share": right_share})
            print("  [%.2f, %.2f%s  %5d syllables, %5.1f%% right" % (
                low, min(high, 1.0), "]" if high > 1 else ")", len(members), 100 * right_share))
    return report


ocr_conf_report = check_ocr_conf()

## 8. Choose tau on val

The corrector's suggestion for a syllable is accepted only when its confidence is at least tau. The sweep covers `TAU_GRID` plus a "no correction" row. For each cap in `CAPS` the rule is: among the settings whose broken rate (Option 2) is at most the cap, take the one with the lowest CER; ties go to the higher tau. If "no correction" wins, the conclusion for that cap is not to correct.

Compare both caps below, then set `FINAL_CAP` in cell 2 by hand and run this cell again to lock tau.

In [ ]:
def sweep(lines):
    rows = []
    for tau in TAU_GRID + [None]:
        pairs = [(line["gt"], final_text(line, tau)) for line in lines]
        cer, wer = cer_wer(pairs)
        cell1 = four_cell(lines, 1, tau)
        cell2 = four_cell(lines, 2, tau)
        rows.append({
            "tau": tau, "cer": cer, "wer": wer,
            "broken_rate_opt2": cell2["broken_rate"], "fixed_opt2": cell2["fixed"], "broken_opt2": cell2["broken"],
            "net_fixed_opt2": cell2["net_fixed"], "broken_rate_opt1": cell1["broken_rate"],
            "net_fixed_opt1": cell1["net_fixed"],
        })
    return rows


def choose_tau(rows, cap):
    candidates = [row for row in rows if row["broken_rate_opt2"] <= cap]
    return min(candidates, key=lambda row: (row["cer"], -(row["tau"] if row["tau"] is not None else 2.0)))


def tau_label(tau):
    return "none" if tau is None else "%.2f" % tau


banner()
sweep_rows = sweep(val_lines)
print("%-6s %8s %8s %13s %11s %11s %13s" % (
    "tau", "CER", "WER", "broken opt2", "net opt2", "broken opt1", "net opt1"))
for row in sweep_rows:
    print("%-6s %7.2f%% %7.2f%% %12.2f%% %11d %10.2f%% %13d" % (
        tau_label(row["tau"]), 100 * row["cer"], 100 * row["wer"], 100 * row["broken_rate_opt2"],
        row["net_fixed_opt2"], 100 * row["broken_rate_opt1"], row["net_fixed_opt1"]))

chosen_by_cap = {}
print("\ncompare the caps:")
print("%-8s %-6s %8s %8s %13s %13s" % ("cap", "tau", "CER", "WER", "broken opt2", "broken opt1"))
for cap in CAPS:
    best = choose_tau(sweep_rows, cap)
    chosen_by_cap[cap] = best
    flag = "  (opt1 breaks the cap)" if best["broken_rate_opt1"] > cap else ""
    print("%-8s %-6s %7.2f%% %7.2f%% %12.2f%% %12.2f%%%s" % (
        "%.0f%%" % (100 * cap), tau_label(best["tau"]), 100 * best["cer"], 100 * best["wer"],
        100 * best["broken_rate_opt2"], 100 * best["broken_rate_opt1"], flag))
    if best["tau"] is None:
        print("   -> at this cap no correction beats correcting: the conclusion is not to correct")

import matplotlib.pyplot as plt

grid_rows = [row for row in sweep_rows if row["tau"] is not None]
figure, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot([row["tau"] for row in grid_rows], [100 * row["cer"] for row in grid_rows], marker="o")
axes[0].set_xlabel("tau")
axes[0].set_ylabel("CER on val (%)")
axes[0].set_title("CER by tau")
axes[1].plot([row["tau"] for row in grid_rows], [100 * row["broken_rate_opt2"] for row in grid_rows], marker="o")
for cap in CAPS:
    axes[1].axhline(100 * cap, linestyle="--", color="gray")
axes[1].set_xlabel("tau")
axes[1].set_ylabel("broken rate, Option 2 (%)")
axes[1].set_title("Broken rate by tau (dashed: caps)")
plt.tight_layout()
plt.show()

locked = None
if FINAL_CAP is None:
    print("FINAL_CAP is not set. Compare the caps above, set FINAL_CAP in cell 2, then run this cell again.")
else:
    assert FINAL_CAP in CAPS, "FINAL_CAP must be one of CAPS"
    best = choose_tau(sweep_rows, FINAL_CAP)
    locked = {
        "cap": FINAL_CAP, "tau": best["tau"], "val_cer": best["cer"], "val_wer": best["wer"],
        "basis": "val, Option 2 broken rate", "suffix": RESULT_SUFFIX,
        "level": CORRECTOR_LEVEL or "default", "commit": GIT_HEAD,
    }
    write_json(f"{OUT_DIR}/eval_tau{RESULT_SUFFIX}.json", locked)

## 9. Bootstrap

Lines of one passage are not independent (many writers copy the same text), so the resampling unit is the **passage**: draw as many passages as there are, with replacement, pool their lines, compute CER before and after, and repeat `N_BOOT` times. The interval is the 2.5% to 97.5% range of the CER drop (before minus after). If the interval contains 0, do not claim an improvement.

The page-level table is a comparison only (val has 30 pages, but pages of one passage share the text). On test the two levels almost coincide.

In [ ]:
def bootstrap(lines, tau, unit):
    by_unit = collections.defaultdict(lambda: [0, 0, 0])
    for line in lines:
        key = line["passage"] if unit == "passage" else line["page"]
        by_unit[key][0] += edits(line["gt"], line["ocr_norm"])
        by_unit[key][1] += edits(line["gt"], final_text(line, tau))
        by_unit[key][2] += len(line["gt"])
    units = sorted(by_unit)
    total_before = sum(by_unit[key][0] for key in units)
    total_after = sum(by_unit[key][1] for key in units)
    total_chars = sum(by_unit[key][2] for key in units)
    rng = random.Random(SEED)
    drops = []
    for _ in range(N_BOOT):
        drawn = rng.choices(units, k=len(units))
        before = sum(by_unit[key][0] for key in drawn)
        after = sum(by_unit[key][1] for key in drawn)
        chars = sum(by_unit[key][2] for key in drawn)
        drops.append((before - after) / chars)
    drops.sort()
    return {
        "unit": unit, "units": len(units),
        "cer_before": total_before / total_chars, "cer_after": total_after / total_chars,
        "drop": (total_before - total_after) / total_chars,
        "ci_low": drops[int(0.025 * N_BOOT)], "ci_high": drops[int(0.975 * N_BOOT) - 1],
        "share_positive": sum(1 for value in drops if value > 0) / N_BOOT,
    }


def boot_configs():
    configs = [("tau=0.00 (correct all)", 0.0)]
    for cap in CAPS:
        configs.append(("cap %.0f%% -> tau %s" % (100 * cap, tau_label(chosen_by_cap[cap]["tau"])), chosen_by_cap[cap]["tau"]))
    return configs


def print_boot(results):
    print("%-26s %6s %9s %9s %9s %20s %8s" % ("setting", "units", "CER before", "CER after", "drop", "95% interval of drop", "drop>0"))
    for name, result in results:
        print("%-26s %6d %8.2f%% %8.2f%% %8.2f%% %9.2f%% to %6.2f%% %7.0f%%" % (
            name, result["units"], 100 * result["cer_before"], 100 * result["cer_after"], 100 * result["drop"],
            100 * result["ci_low"], 100 * result["ci_high"], 100 * result["share_positive"]))


banner()
boot_passage = [(name, bootstrap(val_lines, tau, "passage")) for name, tau in boot_configs()]
boot_page = [(name, bootstrap(val_lines, tau, "page")) for name, tau in boot_configs()]
print("by passage (main):")
print_boot(boot_passage)
print("\nby page (comparison, val only):")
print_boot(boot_page)

## 9b. Compare the corrector levels on val

Runs every level in `CORRECTOR_LEVELS` plus the default `correct()` on the val lines. For each level: contract violations (a level with any is marked INVALID and left out), determinism (the first 100 lines are run twice), time, fixed and broken syllables at tau = 0, the best tau per cap, and the calibration of `conf`.

**Calibration (ECE, 10 equal-width bins, weighted by the number of syllables in each bin).** The event is "the corrected syllable equals the label". It is measured three ways, because the corrector can only be right where the base letters are already right (82% of the OCR errors are base letters, outside its reach):
- *all*: every compared syllable (pessimistic: base-letter errors count as wrong whatever `conf` says);
- *changed*: only syllables the corrector changed (the ones that tau acts on);
- *recoverable*: only syllables whose base letters are right, so a perfect corrector could match the label. This is the fair measure of the corrector's own calibration.

**Rule for the level, written before the numbers (settings, section 2):** at `FINAL_CAP`, the level with the lowest val CER; levels within `TIE_POINTS` CER points of the best are tied, and a tie goes to the simpler level (`SIMPLER_FIRST`). Choosing the level, the cap and tau on the same val split makes the val interval optimistic, so the test split (scored once) is the number to trust.

**Sanity check from theorem B4:** no diacritic-only corrector can lower the CER by more than the exact ceiling of 5d. If a level does, the notebook stops, because the comparison or the contract check is wrong.

In [ ]:
def quantiles(values, qs=(0.1, 0.25, 0.5, 0.75, 0.9)):
    ordered = sorted(values)
    return [ordered[min(int(q * len(ordered)), len(ordered) - 1)] for q in qs] if ordered else []


UNIT_KEY = {"passage": "passage", "page": "page"}


def ece(samples, bins=10):
    """samples: list of (confidence, is_right). Equal-width bins, each weighted by its number of samples."""
    if not samples:
        return float("nan")
    buckets = [[0, 0.0, 0] for _ in range(bins)]
    for confidence, is_right in samples:
        index = min(int(confidence * bins), bins - 1)
        buckets[index][0] += 1
        buckets[index][1] += confidence
        buckets[index][2] += 1 if is_right else 0
    return sum(abs(bucket[1] - bucket[2]) for bucket in buckets if bucket[0]) / len(samples)


def run_level(level, lines_src, check_repeat=100):
    """Run one level on copies of the lines. Returns (lines, violations, seconds, lines that differ when run twice)."""
    started = time.time()
    fn = get_level_fn(level)
    load_seconds = time.time() - started
    lines = [dict(line) for line in lines_src]
    violations = []
    started = time.time()
    for line in lines:
        text_in = line["ocr_norm"]
        try:
            out = fn(text_in)
            problems = contract_checks.check_correct_output(text_in, out)
        except Exception as error:
            out, problems = None, ["raised %s: %s" % (type(error).__name__, error)]
        if problems:
            violations.append((line["id"], problems[:2]))
            continue
        line["corr_syll"] = out["text"].split()
        line["conf"] = [float(value) for value in out["conf"]]
    run_seconds = time.time() - started
    nondeterministic = 0
    if not violations:
        for line in lines[:check_repeat]:
            again = fn(line["ocr_norm"])
            if again["text"].split() != line["corr_syll"] or any(
                    abs(a - b) > 1e-9 for a, b in zip(again["conf"], line["conf"])):
                nondeterministic += 1
    return lines, violations, {"load": load_seconds, "run": run_seconds}, nondeterministic


def changed_stats(lines):
    """Option 2 pairs, tau = 0: how many syllables the corrector changes and what happens to them."""
    counts = collections.Counter()
    for line in lines:
        gt = line["gt"].split()
        ocr = line["ocr_norm"].split()
        corr = line["corr_syll"]
        for ref_pos, hyp_pos in line["pairs2"]:
            if corr[hyp_pos] == ocr[hyp_pos]:
                continue
            counts["changed"] += 1
            if corr[hyp_pos] == gt[ref_pos]:
                counts["to_right"] += 1  # OCR was wrong, now right (fixed)
            elif ocr[hyp_pos] == gt[ref_pos]:
                counts["broke_right"] += 1  # OCR was right, now wrong (broken)
            else:
                counts["wrong_to_other_wrong"] += 1
    return counts


def conf_stats(lines, bins=10):
    """Calibration of the corrector's confidence, three subsets (see the text above)."""
    all_samples, changed_samples, recoverable_samples = [], [], []
    for line in lines:
        gt = line["gt"].split()
        ocr = line["ocr_norm"].split()
        for ref_pos, hyp_pos in line["pairs2"]:
            confidence = line["conf"][hyp_pos]
            is_right = line["corr_syll"][hyp_pos] == gt[ref_pos]
            all_samples.append((confidence, is_right))
            if line["corr_syll"][hyp_pos] != ocr[hyp_pos]:
                changed_samples.append((confidence, is_right))
            if strip_diacritics(ocr[hyp_pos]) == strip_diacritics(gt[ref_pos]):
                recoverable_samples.append((confidence, is_right))
    values = sorted(value for line in lines for value in line["conf"])
    return {
        "bins": bins,
        "ece_all": ece(all_samples, bins), "n_all": len(all_samples),
        "ece_changed": ece(changed_samples, bins), "n_changed": len(changed_samples),
        "ece_recoverable": ece(recoverable_samples, bins), "n_recoverable": len(recoverable_samples),
        "share_conf_ge_095": sum(1 for v in values if v >= 0.95) / len(values) if values else float("nan"),
        "conf_q10_q50_q90": quantiles(values, (0.1, 0.5, 0.9)),
    }


def paired_bootstrap(lines_a, tau_a, lines_b, tau_b, unit="passage"):
    """Bootstrap by passage of CER(a) - CER(b) on the same lines. Positive = b is better."""
    by_unit = collections.defaultdict(lambda: [0, 0, 0])
    for line_a, line_b in zip(lines_a, lines_b):
        assert line_a["id"] == line_b["id"]
        key = line_a[UNIT_KEY[unit]]
        by_unit[key][0] += edits(line_a["gt"], final_text(line_a, tau_a))
        by_unit[key][1] += edits(line_b["gt"], final_text(line_b, tau_b))
        by_unit[key][2] += len(line_a["gt"])
    units = sorted(by_unit)
    rng = random.Random(SEED)
    diffs = []
    for _ in range(N_BOOT):
        drawn = rng.choices(units, k=len(units))
        chars = sum(by_unit[key][2] for key in drawn)
        diffs.append((sum(by_unit[key][0] for key in drawn) - sum(by_unit[key][1] for key in drawn)) / chars)
    diffs.sort()
    total_chars = sum(by_unit[key][2] for key in units)
    return {
        "units": len(units),
        "diff": (sum(by_unit[key][0] for key in units) - sum(by_unit[key][1] for key in units)) / total_chars,
        "ci_low": diffs[int(0.025 * N_BOOT)], "ci_high": diffs[int(0.975 * N_BOOT) - 1],
    }


def compare_levels():
    if USE_FAKE:
        print("Skipped: USE_FAKE is True, there are no real levels to compare.")
        return "not run"
    banner()
    level_runs, level_report = {}, {}
    for level in [None] + list(CORRECTOR_LEVELS):
        name = level or "default"
        try:
            lines_l, violations, seconds, nondeterministic = run_level(level, val_lines)
        except Exception as error:  # for example the tagger has no checkpoint yet
            level_report[name] = {"status": "unavailable", "error": "%s: %s" % (type(error).__name__, error)}
            print("%-10s unavailable: %s" % (name, level_report[name]["error"]))
            continue
        if violations:
            level_report[name] = {"status": "contract violations", "count": len(violations), "examples": violations[:5]}
            print("%-10s INVALID: %d of %d lines break the contract, for example %s" % (
                name, len(violations), len(val_lines), violations[:2]))
            continue
        rows = sweep(lines_l)
        level_runs[name] = lines_l
        level_report[name] = {
            "status": "ok", "seconds": seconds, "nondeterministic_lines": nondeterministic, "sweep": rows,
            "tau0_opt2": four_cell(lines_l, 2, 0.0), "changed_tau0": dict(changed_stats(lines_l)),
            "confidence": conf_stats(lines_l),
            "chosen_by_cap": {str(cap): choose_tau(rows, cap) for cap in CAPS},
        }

    valid = [name for name, entry in level_report.items() if entry["status"] == "ok"]
    assert valid, "no corrector level passed the contract check on val"
    print("\ncontract: 0 violations on %d val lines for: %s" % (len(val_lines), ", ".join(valid)))

    print("\n%-10s %7s %7s | %6s %6s %8s %9s | %8s %9s %11s" % (
        "level", "load s", "run s", "fixed", "broken", "changed", "precision", "ECE all", "ECE chgd", "ECE recov."))
    for name in valid:
        entry = level_report[name]
        changed = entry["changed_tau0"].get("changed", 0)
        precision = entry["changed_tau0"].get("to_right", 0) / changed if changed else float("nan")
        stats = entry["confidence"]
        print("%-10s %7.1f %7.1f | %6d %6d %8d %8.1f%% | %8.3f %9.3f %11.3f" % (
            name, entry["seconds"]["load"], entry["seconds"]["run"], entry["tau0_opt2"]["fixed"],
            entry["tau0_opt2"]["broken"], changed, 100 * precision,
            stats["ece_all"], stats["ece_changed"], stats["ece_recoverable"]))
        if entry["nondeterministic_lines"]:
            print("   WARNING: %d of the first 100 lines gave a different output when run twice;"
                  " tau chosen on val would not be reproducible" % entry["nondeterministic_lines"])
        print("   conf: %.1f%% of syllables have conf >= 0.95; q10/q50/q90 = %s" % (
            100 * stats["share_conf_ge_095"], ", ".join("%.2f" % v for v in stats["conf_q10_q50_q90"])))

    exact_limit = exact_ceiling_report["cer_ceiling_points"] if isinstance(exact_ceiling_report, dict) else None
    print("\nbest setting per level and cap (val CER without correction %.2f%%; exact ceiling of any diacritic-only corrector: %s)" % (
        100 * cer_norm, "%.2f points" % (100 * exact_limit) if exact_limit is not None else "run section 5d first"))
    print("%-10s %-5s %-5s %7s %7s %12s %9s %24s" % (
        "level", "cap", "tau", "CER", "drop", "broken opt2", "of ceil.", "95% interval of drop"))
    boot_by_level = {}
    for name in valid:
        for cap in CAPS:
            best = level_report[name]["chosen_by_cap"][str(cap)]
            boot = bootstrap(level_runs[name], best["tau"], "passage")
            boot_by_level[(name, cap)] = boot
            if exact_limit is not None:  # theorem B4
                assert cer_norm - best["cer"] <= exact_limit + 1e-9, (
                    "%s: CER drop %.4f is above the exact ceiling %.4f, so the comparison or the contract check is wrong" % (
                        name, cer_norm - best["cer"], exact_limit))
            share = ("%8.0f%%" % (100 * (cer_norm - best["cer"]) / exact_limit)) if exact_limit else "      n/a"
            print("%-10s %-5s %-5s %6.2f%% %6.2f%% %11.2f%% %s %10.2f%% to %5.2f%%" % (
                name, "%.0f%%" % (100 * cap), tau_label(best["tau"]), 100 * best["cer"], 100 * (cer_norm - best["cer"]),
                100 * best["broken_rate_opt2"], share, 100 * boot["ci_low"], 100 * boot["ci_high"]))
            if best["tau"] is None:
                print("   -> no correction beats not correcting at this cap: the result for this level is: do not correct")

    named = [name for name in valid if name != "default"]
    if "default" in valid:
        for name in named:
            if all(a["corr_syll"] == b["corr_syll"] and a["conf"] == b["conf"]
                   for a, b in zip(level_runs["default"], level_runs[name])):
                print("\ndefault correct() gives exactly the same text and conf as level %r on val" % name)

    if len(named) >= 2:
        print("\npaired comparison (CER of A minus CER of B, each at its own best tau; positive = B is better):")
        for cap in CAPS:
            for index_a, name_a in enumerate(named):
                for name_b in named[index_a + 1:]:
                    tau_a = level_report[name_a]["chosen_by_cap"][str(cap)]["tau"]
                    tau_b = level_report[name_b]["chosen_by_cap"][str(cap)]["tau"]
                    result = paired_bootstrap(level_runs[name_a], tau_a, level_runs[name_b], tau_b)
                    verdict = ("interval contains 0: no clear winner" if result["ci_low"] <= 0 <= result["ci_high"]
                               else "%s is better" % (name_b if result["diff"] > 0 else name_a))
                    print("  cap %.0f%%: %-9s vs %-9s diff %6.3f points, interval %6.3f to %6.3f -> %s" % (
                        100 * cap, name_a, name_b, 100 * result["diff"], 100 * result["ci_low"], 100 * result["ci_high"], verdict))

    if FINAL_CAP is not None and named:
        ranked = sorted(named, key=lambda name: level_report[name]["chosen_by_cap"][str(FINAL_CAP)]["cer"])
        best_cer = level_report[ranked[0]]["chosen_by_cap"][str(FINAL_CAP)]["cer"]
        tied = [name for name in ranked
                if 100 * (level_report[name]["chosen_by_cap"][str(FINAL_CAP)]["cer"] - best_cer) <= TIE_POINTS]
        suggestion = min(tied, key=lambda name: SIMPLER_FIRST.index(name) if name in SIMPLER_FIRST else 99)
        print("\nrule written before the numbers: lowest val CER at FINAL_CAP=%.0f%%, ties within %.2f points go to the simpler level" % (
            100 * FINAL_CAP, TIE_POINTS))
        print("-> suggested level: %r. Set CORRECTOR_LEVEL in the settings, run again from section 6a, then lock tau." % suggestion)
    else:
        print("\nSet FINAL_CAP in the settings to see the suggested level.")

    # the main flow (sections 6 to 8) and this comparison must agree for the level used in the main flow
    main_name = CORRECTOR_LEVEL or "default"
    if main_name in level_report and level_report[main_name]["status"] == "ok":
        for cap in CAPS:
            mine = level_report[main_name]["chosen_by_cap"][str(cap)]
            theirs = chosen_by_cap[cap]
            assert mine["tau"] == theirs["tau"] and abs(mine["cer"] - theirs["cer"]) < 1e-12, (
                "the main flow and the comparison disagree for %s at cap %s: run again from section 6a" % (main_name, cap))
        print("check: the main flow and the comparison agree for level %r" % main_name)

    report = dict(level_report)
    report["_bootstrap_passage"] = {"%s|%s" % (name, cap): boot for (name, cap), boot in boot_by_level.items()}
    return report


level_report_json = compare_levels()

## 10. Score the test split (once)

Off by default. It runs only when all of these hold: `RUN_TEST = True`, the OCR file is final and the corrector is the real one (no `_DRAFT` or `_FAKE` suffix), `eval_tau.json` exists, and `eval_test_result.json` does not exist yet. After the run the result file blocks any second run. The OCR file must contain all test lines of `splits.json`.

In [ ]:
TAU_PATH = f"{OUT_DIR}/eval_tau.json"
TEST_RESULT_PATH = f"{OUT_DIR}/eval_test_result.json"


def run_test():
    if not RUN_TEST:
        print("RUN_TEST is False: the test split is not scored.")
        return
    if RESULT_SUFFIX:
        print("refused: the output is marked %s. Use the final OCR file and the real corrector first." % RESULT_SUFFIX)
        return
    if not os.path.exists(TAU_PATH):
        print("refused: lock tau on val first (eval_tau.json is missing).")
        return
    if os.path.exists(TEST_RESULT_PATH):
        print("refused: the test split was already scored once (eval_test_result.json exists).")
        return
    locked_info = read_json(TAU_PATH)
    if locked_info.get("level") != (CORRECTOR_LEVEL or "default"):
        print("refused: tau was locked for level %r but CORRECTOR_LEVEL is %r. Run again from section 6a." % (
            locked_info.get("level"), CORRECTOR_LEVEL or "default"))
        return
    if locked_info.get("commit") != GIT_HEAD:
        print("warning: the repo commit changed since tau was locked (%s -> %s). If the model files changed, tau is no longer valid." % (
            locked_info.get("commit"), GIT_HEAD))
    tau = locked_info["tau"]
    test_lines = build_lines(read_jsonl(OCR_OUT_PATH), "test")
    correct_lines(test_lines)
    prepare_pairs(test_lines)
    pairs_before_raw = [(line["gt"], line["ocr_raw"]) for line in test_lines]
    pairs_before_norm = [(line["gt"], line["ocr_norm"]) for line in test_lines]
    pairs_after = [(line["gt"], final_text(line, tau)) for line in test_lines]
    result = {
        "tau": tau,
        "level": CORRECTOR_LEVEL or "default",
        "commit": GIT_HEAD,
        "lines": len(test_lines),
        "cer_wer_before_raw": cer_wer(pairs_before_raw),
        "cer_wer_before_normalized": cer_wer(pairs_before_norm),
        "cer_wer_after": cer_wer(pairs_after),
        "table_option1": four_cell(test_lines, 1, tau),
        "table_option2": four_cell(test_lines, 2, tau),
        "bootstrap_passage": bootstrap(test_lines, tau, "passage"),
    }
    for key, value in result.items():
        print(key, value)
    write_json(TEST_RESULT_PATH, result)


run_test()

## 11. Save the val results to Drive

In [ ]:
summary = {
    "suffix": RESULT_SUFFIX,
    "ocr_out_path": OCR_OUT_PATH,
    "val_lines": len(val_lines),
    "settings": {
        "caps": CAPS, "final_cap": FINAL_CAP, "n_boot": N_BOOT, "seed": SEED,
        "p_min": P_MIN, "ceiling_min": CEILING_MIN, "use_fake": USE_FAKE, "ocr_is_draft": OCR_IS_DRAFT,
    },
    "ocr_quality": {
        "cer_raw": cer_raw, "wer_raw": wer_raw, "cer_normalized": cer_norm, "wer_normalized": wer_norm,
        "empty_lines": empty_lines, "same_syllable_count_lines": same_count,
    },
    "corrector": {"level": CORRECTOR_LEVEL or "default", "module": CORRECTOR_MODULE, "commit": GIT_HEAD,
                  "use_fake": USE_FAKE, "levels_compared": CORRECTOR_LEVELS},
    "gate": gate_report,
    "exploratory_syllable_level": globals().get("exploratory_report", "not run"),
    "exact_ceiling": globals().get("exact_ceiling_report", "not run"),
    "conf_distribution": globals().get("conf_distribution", "not run"),
    "ocr_conf_check": globals().get("ocr_conf_report", "not run"),
    "level_comparison": globals().get("level_report_json", "not run"),
    "table_tau0": {"option1": table_tau0[1], "option2": table_tau0[2]},
    "sweep": sweep_rows,
    "chosen_by_cap": {str(cap): row for cap, row in chosen_by_cap.items()},
    "locked": locked,
    "bootstrap_passage": [{"setting": name, **result} for name, result in boot_passage],
    "bootstrap_page": [{"setting": name, **result} for name, result in boot_page],
}
banner()
write_json(f"{OUT_DIR}/eval_val{RESULT_SUFFIX}.json", summary)